# SHAPxAdopt-GH: run the full experiment
Run each cell **in order**: click the ▶ button on its left, or press **Shift + Enter**, and wait for it to finish before starting the next one.

| Step | What it does | Time |
|---|---|---|
| 1 | Upload your files | 1 min |
| 2 | Install the packages | 1–2 min |
| 3 | Run the full experiment (every table and figure in Chapter 4 and the appendices) | 3–6 min |
| 4 | Show the key results | seconds |
| 5 | Show the figures | seconds |
| 6 | Run the prediction script (`predict.py`) | 30 s |
| 7 | Download all outputs as one zip | seconds |

## Step 1: Upload your files
When the **Choose Files** button appears, select these **four files** together (hold **Ctrl**):
- `shapxadopt_revision_experiments.py`
- `predict.py`
- `requirements.txt`
- your data workbook, `AI_Readiness_Cleaned_Workbook_300_1.xlsx`

In [ ]:
from google.colab import files
import glob, os
uploaded = files.upload()

# tidy names: if a browser saved "predict (1).py", rename it back
for f in list(os.listdir('.')):
    for good in ['predict.py', 'shapxadopt_revision_experiments.py', 'requirements.txt']:
        stem, ext = os.path.splitext(good)
        if f != good and f.startswith(stem) and f.endswith(ext) and not os.path.exists(good):
            os.rename(f, good); print(f'renamed {f} -> {good}')

need = ['shapxadopt_revision_experiments.py', 'predict.py']
missing = [f for f in need if not os.path.exists(f)]
xlsx = sorted(glob.glob('*.xlsx'))
print('Python files found:', [f for f in need if os.path.exists(f)])
print('Workbook found    :', xlsx)
if missing: print('MISSING:', missing, '-> run this cell again and add them')
if not xlsx: print('MISSING: the .xlsx workbook -> run this cell again and add it')
DATA = xlsx[0] if xlsx else None

## Step 2: Install the packages
Most packages are already in Colab; this adds the few that are missing and prints the versions used.
If Colab shows a **Restart session** button after this cell, click it, then carry on with Step 3. Your uploaded files stay in place.

In [ ]:
!pip -q install xgboost shap lightgbm imbalanced-learn statsmodels openpyxl scikit-learn
import sys, xgboost, shap, sklearn, lightgbm, imblearn, pandas, numpy
print('Python', sys.version.split()[0])
for m in (xgboost, shap, sklearn, lightgbm, imblearn, pandas, numpy):
    print(f'{m.__name__:10s} {m.__version__}')

## Step 3: Run the full experiment
This repeats everything in the thesis: the five models, 5-fold cross-validation, 10 seeds, the ablations, leave-one-tier-out, the noise test, grouped cross-validation, the fairness audit and SHAP.
You will see `STEP 0` … `STEP 11 done` and finally `All steps done`.

In [ ]:
import glob
DATA = sorted(glob.glob('*.xlsx'))[0]   # your workbook from Step 1
print('Using', DATA)
!python shapxadopt_revision_experiments.py --data "$DATA" 

## Step 4: Key results
Compare these with the thesis. The thesis numbers are shown above each table.
If Colab has newer package versions than the thesis (see Step 2), a few numbers may differ in the second or third decimal place; that is normal.

In [ ]:
import pandas as pd, json, os
from IPython.display import display, Markdown
O = 'revision_outputs'
def show(title, file, **kw):
    display(Markdown(f'**{title}**'))
    display(pd.read_csv(f'{O}/{file}', **kw))

show('Table 4.1: test-set performance (thesis: XGBoost F1 0.894, AUC-ROC 0.979)', 'S3_test_metrics.csv')
show('Table 4.2: McNemar tests (thesis: all p = 1.000)', 'S3_mcnemar.csv')
show('5-fold cross-validation, mean ± SD', 'S1_cv_mean_sd.csv')
show('Ten random seeds, mean ± SD', 'S2_seed_mean_sd.csv')
show('Table 4.3: ablations and sensitivity, mean ± SD over 10 seeds', 'S4_S7_S8_S9_mean_sd.csv')
show('Leave-one-tier-out', 'S5_leave_one_tier_out.csv')
show('Noise (perturbation) test', 'S6_perturbation.csv')
show('Table 4.4: performance by institution tier', 'S10_per_tier.csv')
show('Grouped (duplicate-aware) cross-validation', 'S14_grouped_cv_mean_sd.csv')
show('Equalised-odds audit (all 300, out-of-fold)', 'S15_equalised_odds_by_tier.csv')
display(Markdown('**Run log** (model complexity, duplicates, etc.)'))
print(json.dumps(json.load(open(f'{O}/run_log.json')), indent=2))

## Step 5: Figures

In [ ]:
from IPython.display import Image
for f in ['Figure5_stability_boxplot.png', 'Figure6a_shap_importance.png', 'Figure6b_shap_beeswarm_high.png',
          'FigureB3_shap_vs_gain.png', 'FigureB4_waterfall_Low.png', 'FigureB4_waterfall_Medium.png', 'FigureB4_waterfall_High.png']:
    p = f'{O}/{f}'
    if os.path.exists(p):
        display(Markdown(f'**{f}**')); display(Image(p, width=800))

## Step 6: The prediction script
Trains the primary model, then predicts the 45 held-out test cases and lists the three features behind each prediction.

In [ ]:
import glob, pandas as pd
DATA = sorted(glob.glob('*.xlsx'))[0]
!python predict.py train --data "$DATA"
!python predict.py predict
from sklearn.metrics import f1_score
import shapxadopt_revision_experiments as S
p = pd.read_csv('predictions.csv'); d = pd.read_csv('demo_test_cases.csv')
y = S.make_target(d['C5'].values, S.THRESHOLDS_MAIN)
pred = p.predicted_tier.map({'Low': 0, 'Medium': 1, 'High': 2}).values
print('\nMacro F1 on the 45 test cases:', round(f1_score(y, pred, average='macro'), 3), '(thesis: 0.894)')

## Step 7: Download everything

In [ ]:
from google.colab import files
!zip -q -r revision_outputs.zip revision_outputs predictions.csv
files.download('revision_outputs.zip')